# Wine Quality EDA
<a id='0'></a>
Table of Contents

1. <a href='#1'>Introduction</a>
2. <a href='#2'>Data Loading and Overview</a>
3. <a href='#3'>Red vs White: Fundamental Differences</a>
4. <a href='#4'>Correlation Analysis</a>
5. <a href='#5'>Derived Features</a>
6. <a href='#6'>Feature Distributions by Quality</a>
7. <a href='#7'>Multivariate Exploration</a>
8. <a href='#8'>Results and Discussion</a>
   - Universal findings
   - Red-specific findings
   - White-specific findings
9. <a href='#9'>Limitations</a>
10. <a href='#10'>Conclusions</a>



## 1. Introduction
<a id='1'></a>
**Goal:** Explore the relationship between composition, physicochemical properties and wine quality ratings.

**Dataset:** UCI Wine Quality (red wines, n=1599, white wines, n=4898)  
**Source:** https://archive.ics.uci.edu/ml/datasets/wine+quality

**Key questions:**
- Which chemical parameters correlate most strongly with quality?
- Are there natural types of wine compositions which reveals itselves in clustering?
- Can we separate "good" (quality ≥ 7) or bad (quality<5) from "average" wines?

In [ ]:
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from pandas.plotting import parallel_coordinates
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.mixture import GaussianMixture


import warnings
warnings.filterwarnings('ignore')



## 2. Data Loading and Overview <a id='2'></a>

In [ ]:
red_file = 'winequality-red.csv'
white_file = 'winequality-white.csv'
red_wines = pd.read_csv(red_file, sep=';')
white_wines = pd.read_csv(white_file, sep=';')
# Loading both red and white wines datasets and combine them in one
red_wines['type'] = 'red'
white_wines['type'] = 'white'
df_raw = pd.concat([red_wines, white_wines], ignore_index=True) #This dataframe contains raw data for both red and white wines
# Further we are going to get some derived features in separate dataframe


In [ ]:
df_raw.info()
df_raw.head()
df_raw.tail()

In [ ]:
###-------FUNCTIONS DEFINITION------###

def classify_values(series, typical_low, typical_high, absolute_low, absolute_high):
    """Classifies values as typical, atypical or physically impossible"""
    conditions = [
        (series < absolute_low) | (series > absolute_high),
        (series < typical_low) | (series > typical_high),
    ]
    choices = ['physically_impossible', 'atypical']
    return np.select(conditions, choices, default='typical')

# Code below converts wine composition in g/L used in original dataset to mol/L. Molar concentration is more natural and it is easier to make derived features that make sense
molar_masses = {
    # Not all of them are actually molar masses, some columns copied to new dataframe without conversion simply multiplied by 1
    'fixed acidity': 150.09, #tartaric acid
    'volatile acidity': 60.05, #acetic acid
    'citric acid': 210.14, 
    'residual sugar': 180.16, #glucose, fructose
    'chlorides': 58.44, #sodium chloride
    'free sulfur dioxide': 64.06,
    'total sulfur dioxide': 64.06,
    'density': 1,
    'pH': 1,
    'sulphates': 174.25, #potassium sulphate
    'alcohol': 1,
    'quality': 1
        }
#--Converting concentrations in grams per liter to molar concentrations--
def molar_c(gram_liter, component_name) :
    molar_mass = molar_masses[component_name]
    return gram_liter/molar_mass

def plot_correlated_pairs(df_subset, wine_type_name, threshold=0.3):
    """Draws scatter plots for pairs with |ρ| > threshold in selected subdataset."""
    
    # Calculate correlations for exact type only
    cols = numeric_cols + ['quality']
    corr_matrix = df_subset[cols].corr(method='spearman')
    
    # Find pairs of correlating features
    pairs = []
    for i in range(len(corr_matrix.columns)):
        for j in range(i + 1, len(corr_matrix.columns)):
            col1 = corr_matrix.columns[i]
            col2 = corr_matrix.columns[j]
            r_value = corr_matrix.iloc[i, j]
            if abs(r_value) > threshold:
                pairs.append((col1, col2, r_value))
    
    pairs.sort(key=lambda x: abs(x[2]), reverse=True)
    print(f"{wine_type_name}: {len(pairs)} pairs with |ρ| > {threshold} found")
    
    if len(pairs) == 0:
        return
    
    # GRID
    n_pairs = len(pairs)
    n_grid_cols = 4
    n_grid_rows = (n_pairs + n_grid_cols - 1) // n_grid_cols
    
    fig, axes = plt.subplots(n_grid_rows, n_grid_cols, 
                             figsize=(18, 4.5 * n_grid_rows))
    if n_grid_rows == 1:
        axes = np.array([axes])
    axes = axes.flatten()
    
    for idx, (col1, col2, r_value) in enumerate(pairs):
        ax = axes[idx]
        
        scatter = ax.scatter(
            df_subset[col1], df_subset[col2], 
            c=df_subset['quality'], 
            cmap='viridis', 
            alpha=0.6, 
            s=5  
        )
        
        title_col1 = name_formatting.get(col1, col1)
        title_col2 = name_formatting.get(col2, col2)
        ax.set_title(
            f'{title_col1} vs {title_col2}\nρ = {r_value:.3f}',
            fontsize=10,
            fontweight='bold'
        )
        ax.set_xlabel(title_col1, fontsize=9)
        ax.set_ylabel(title_col2, fontsize=9)
        ax.tick_params(axis='both', labelsize=8)
        ax.grid(True, alpha=0.3)
    
    # removing empty subplots
    for j in range(idx + 1, len(axes)):
        fig.delaxes(axes[j])
    
    plt.suptitle(
        f'{wine_type_name}: Scatter Plots with |ρ| > {threshold} (Spearman)', 
        fontsize=16, 
        fontweight='bold', 
        y=0.998
    )
    plt.tight_layout()
    
    # Colorbar for quality
    if n_pairs > 0:
        fig.colorbar(scatter, ax=axes[:n_pairs], label='Quality', shrink=0.6)
    
    plt.show()
    
    return pairs

##################################################################################    

def plot_quality_comparison(df_all, features):
    """Compares feature distributions by quality for red and white wines."""
    
    n_features = len(features)
    fig, axes = plt.subplots(n_features, 2, figsize=(16, 4 * n_features))
    
    for row, feat in enumerate(features):
        pretty_feat = name_formatting.get(feat, feat.replace('_', ' ').title())
        
        for col, wine_type in enumerate(['red', 'white']):
            ax = axes[row, col]
            df_subset = df_all[df_all['type'] == wine_type]
            
            sns.boxplot(
                x='quality group', 
                y=feat, 
                data=df_subset, 
                ax=ax, 
                palette='Set2'
            )
            
            # Title
            if row == 0:
                type_label = 'Red' if wine_type == 'red' else 'White'
                ax.set_title(f'{type_label} Wines', fontsize=13, fontweight='bold')
            
            # Labels
            if col == 0:
                ax.set_ylabel(pretty_feat, fontsize=10, fontweight='bold')
            else:
                ax.set_ylabel('')
            
            if row < n_features - 1:
                ax.set_xlabel('')
            
            ax.grid(True, alpha=0.3, axis='y')
    
    plt.suptitle(
        'Feature Distributions by Quality Group: Red vs White', 
        fontsize=16, 
        fontweight='bold', 
        y=1.02
    )
    plt.tight_layout()
    plt.show()

#########################################################################################
    
def plot_parallel_coordinates(df_subset, wine_type_name, features):
    """Draws parallel coordinates for exact type of wine with separate normalization."""
    
    # Normalize subset, not whole dataset
    df_norm = df_subset[features].copy()
    df_norm = (df_norm - df_norm.min()) / (df_norm.max() - df_norm.min())
    df_norm['quality group'] = df_subset['quality group'].values
    
    plt.figure(figsize=(14, 7))
    parallel_coordinates(
        df_norm, 
        'quality group', 
        colormap='viridis', 
        alpha=0.4
    )
    plt.title(f'{wine_type_name}: Wine Profiles (Parallel Coordinates)', 
              fontsize=14, fontweight='bold')
    plt.ylabel('Normalized Value')
    plt.grid(True, alpha=0.3)
    plt.show()

################################################################################
    
def cluster_wines(df_subset, wine_type_name, features, n_clusters=4):
    """Clusterize wines of one type and show visualization of the result."""
    
    # Normalize subset
    X = StandardScaler().fit_transform(df_subset[features])
    
    # Clusterize
    kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
    df_subset = df_subset.copy()
    df_subset['cluster'] = kmeans.fit_predict(X)
    
    # Save centroids
    centroids = pd.DataFrame(
        kmeans.cluster_centers_,
        columns=[f'{f}_scaled' for f in features],
        index=[f'Cluster {i}' for i in range(n_clusters)]
    )
    
    return df_subset, kmeans, centroids

######################################################################################
    
def plot_clusters_scatter(df_subset, wine_type_name, features, n_clusters=4):
    """Draws scatterplots for feature pairs with coloured clusters."""
    
    # Feature pairs
    pairs = [
        (features[0], features[1]),  # fermentation vs buffer
        (features[2], features[3]),  # SO2 binding vs volatile fraction
        (features[0], features[3]),  # fermentation vs volatile fraction
        (features[1], features[2]),  # buffer vs SO2 binding
    ]
    
    fig, axes = plt.subplots(2, 2, figsize=(16, 14))
    axes = axes.flatten()
    
    # Palette for visualization
    cluster_palette = sns.color_palette('viridis', n_clusters)
    
    for ax, (feat_x, feat_y) in zip(axes, pairs):
        scatter = ax.scatter(
            df_subset[feat_x], 
            df_subset[feat_y],
            c=df_subset['cluster'],
            cmap='viridis',
            alpha=0.6,
            s=30,
            edgecolors='white',
            linewidth=0.3
        )
        
        pretty_x = name_formatting.get(feat_x, feat_x)
        pretty_y = name_formatting.get(feat_y, feat_y)
        
        ax.set_xlabel(pretty_x, fontsize=10)
        ax.set_ylabel(pretty_y, fontsize=10)
        ax.set_title(f'{pretty_x} vs {pretty_y}', fontsize=11, fontweight='bold')
        ax.grid(True, alpha=0.3)
    
    # Common legend
    plt.suptitle(
        f'{wine_type_name}: Clusters (k={n_clusters})', 
        fontsize=16, fontweight='bold', y=1.02
    )
    
    # Colorbar for clusters
    fig.colorbar(scatter, ax=axes[:len(pairs)], 
                 label='Cluster', shrink=0.6)
    
    plt.tight_layout()
    plt.show()

##################################################################################################

def plot_centroids_fixed(kmeans, scaler, features, wine_type_name, n_clusters=4):
        
    # denormalize
    centroids_original = pd.DataFrame(
        scaler.inverse_transform(kmeans.cluster_centers_),
        columns=features,
        index=[f'Cluster {i}' for i in range(n_clusters)]
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    centroids_norm = centroids_original.copy()
    centroids_norm = (centroids_norm - centroids_norm.min()) / \
                     (centroids_norm.max() - centroids_norm.min())
    centroids_norm['cluster'] = [f'C{i}' for i in range(n_clusters)]
    
    parallel_coordinates(
        centroids_norm, 
        'cluster',
        colormap='viridis', 
        alpha=0.9,
        ax=axes[0]
    )
    axes[0].set_title(f'{wine_type_name}: Cluster Centroids Profiles', 
                     fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Normalized Value')
    axes[0].set_xlabel('Features')
    axes[0].grid(True, alpha=0.3)
    
    sns.heatmap(
        centroids_original, 
        annot=True, 
        fmt='.3f', 
        cmap='viridis',
        ax=axes[1],
        linewidths=0.5
    )
    axes[1].set_title(f'{wine_type_name}: Cluster Centroids (Original Scale)', 
                     fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return centroids_original

def plot_centroids(centroids, features, wine_type_name, n_clusters=4):
    """Visualize centroid of clusters."""
    
    # Denormilize centers
    scaler = StandardScaler()
    scaler.fit(df[df['type'] == wine_type_name.lower().split()[0]][features])
    centroids_original = pd.DataFrame(
        scaler.inverse_transform(centroids.values),
        columns=features,
        index=centroids.index
    )
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1: Centroid parallel coordinates
    centroids_norm = (centroids_original - centroids_original.min()) / \
                     (centroids_original.max() - centroids_original.min())
    centroids_norm['cluster'] = centroids_original.index
    
    parallel_coordinates(
        centroids_norm, 'cluster',
        colormap='viridis', alpha=0.8
    )
    axes[0].set_title(f'{wine_type_name}: Cluster Centroids Profiles', 
                     fontsize=12, fontweight='bold')
    axes[0].set_ylabel('Normalized Value')
    
    # Centroid heatmap
    sns.heatmap(
        centroids_original, 
        annot=True, 
        fmt='.2f', 
        cmap='viridis',
        ax=axes[1]
    )
    axes[1].set_title(f'{wine_type_name}: Cluster Centroids (Original Scale)', 
                     fontsize=12, fontweight='bold')
    
    plt.tight_layout()
    plt.show()
    
    return centroids_original

#################################################################################################

def plot_cluster_quality_distribution(df_subset, wine_type_name, n_clusters=4):
    """Shows quality distribution in each cluster."""
    
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # 1: violin plots
    sns.violinplot(
        x='cluster', 
        y='quality', 
        data=df_subset, 
        ax=axes[0],
        palette='viridis',
        alpha=0.7
    )
    axes[0].set_title(f'{wine_type_name}: Quality Distribution by Cluster', 
                     fontsize=12, fontweight='bold')
    axes[0].set_xlabel('Cluster')
    axes[0].set_ylabel('Quality Score')
    
    # 2: quality groups percentage
    cluster_quality = pd.crosstab(
        df_subset['cluster'], 
        df_subset['quality group'], 
        normalize='index'
    ) * 100
    
    cluster_quality.plot(
        kind='barh', 
        stacked=True, 
        ax=axes[1],
        colormap='Set2',
        alpha=0.8
    )
    axes[1].set_title(f'{wine_type_name}: Quality Composition by Cluster', 
                     fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Percentage')
    axes[1].set_ylabel('Cluster')
    axes[1].legend(title='Quality Group')
    
    plt.tight_layout()
    plt.show()
###############################################################################################
def analyze_clusters(df_subset, wine_type_name, cluster_col, features):
    """Ultimate cluster analysis: stats, quality, visual."""
    
    print(f"\n{'='*60}")
    print(f"{wine_type_name} WINES - CLUSTER ANALYSIS")
    print(f"{'='*60}\n")
    
    print("1. Cluster Sizes:")
    sizes = df_subset[cluster_col].value_counts().sort_index()
    print(sizes)
    print()
    
    print("2. Mean Feature Values by Cluster:")
    all_numeric = [c for c in df_subset.select_dtypes(include=[np.number]).columns 
                   if c not in [cluster_col, 'quality']]
    profiles = df_subset.groupby(cluster_col)[all_numeric].mean()
    print(profiles.round(3))
    print()
    
    print("3. Mean Quality by Cluster:")
    quality_mean = df_subset.groupby(cluster_col)['quality'].mean()
    print(quality_mean.round(2))
    print()
    
    print("4. Quality Group Distribution by Cluster (%):")
    quality_dist = pd.crosstab(
        df_subset[cluster_col], 
        df_subset['quality group'], 
        normalize='index'
    ) * 100
    print(quality_dist.round(1))
    print()
    
    fig, axes = plt.subplots(1, 2, figsize=(18, 6))
    
    profiles_norm = (profiles - profiles.min()) / (profiles.max() - profiles.min())
    
    sns.heatmap(
        profiles_norm, 
        annot=True, 
        fmt='.2f', 
        cmap='viridis',
        ax=axes[0],
        linewidths=0.5
    )
    axes[0].set_title(f'{wine_type_name}: Normalized Cluster Profiles', 
                     fontsize=12, fontweight='bold')
    
    quality_dist.plot(
        kind='barh', 
        stacked=True, 
        ax=axes[1],
        colormap='Set2',
        alpha=0.8
    )
    axes[1].set_title(f'{wine_type_name}: Quality Composition by Cluster', 
                     fontsize=12, fontweight='bold')
    axes[1].set_xlabel('Percentage')
    axes[1].set_ylabel('Cluster')
    axes[1].legend(title='Quality Group')
    
    plt.tight_layout()
    plt.show()
    
    return profiles, quality_dist

##################################################################################################

def plot_all_features_by_cluster(df_subset, wine_type_name, cluster_col, n_clusters=4):
    """Boxplots for all features"""
    
    all_features = [c for c in df_subset.select_dtypes(include=[np.number]).columns 
                    if c not in [cluster_col, 'quality']]
    
    n_features = len(all_features)
    n_cols = 4
    n_rows = (n_features + n_cols - 1) // n_cols
    
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(20, 4 * n_rows))
    axes = axes.flatten()
    
    for ax, feat in zip(axes, all_features):
        sns.boxplot(
            x=cluster_col, 
            y=feat, 
            data=df_subset, 
            ax=ax, 
            palette='viridis'
        )
        pretty_feat = name_formatting.get(feat, feat.replace('_', ' ').title())
        ax.set_title(pretty_feat, fontsize=10, fontweight='bold')
        ax.set_xlabel('Cluster')
        ax.set_ylabel('')
    
    # remove empty
    for j in range(len(all_features), len(axes)):
        fig.delaxes(axes[j])
    
    plt.suptitle(
        f'{wine_type_name}: All Features by Cluster', 
        fontsize=16, fontweight='bold', y=1.02
    )
    plt.tight_layout()
    plt.show()
######################################################################################################
def collinear_groups(feature_list, corr_matrix, threshold=0.7):
    """group features with |r| > threshold."""
    corr = corr_matrix.loc[feature_list, feature_list].abs()
    upper = corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
    
    groups = []
    used = set()
    for col in upper.columns:
        if col in used:
            continue
        correlated = [c for c in upper.index if upper.loc[c, col] > threshold]
        group = [col] + correlated
        groups.append(group)
        used.update(group)
    return groups

######################################################################################################

name_formatting = {
    'pH': 'pH',  
    'fixed acidity': 'Fixed Acidity',
    'volatile acidity': 'Volatile Acidity',
    'citric acid': 'Citric Acid',
    'residual sugar': 'Residual Sugar',
    'chlorides': 'Chlorides',
    'free sulfur dioxide': 'Free Sulfur Dioxide',
    'total sulfur dioxide': 'Total Sulfur Dioxide',
    'density': 'Density',
    'sulphates': 'Sulphates',
    'alcohol': 'Alcohol',
}


## Uncertainty Estimation for Fermentation Completeness

# Measurement uncertainties (typical for OIML/OIV certification methods)
sigma_a = 0.15   # % vol, alcohol
sigma_s = 0.3    # g/L, residual sugar
sigma_v = 0.02   # g/L, volatile acidity

# Constants
RHO_ETHANOL = 0.789   # g/mL
MW_RATIO = 46.07 / 60.05  # ethanol/acetic acid molar mass ratio

def fermentation_completeness_with_uncertainty(a, s, v, 
                                                sigma_a=0.15, 
                                                sigma_s=0.3, 
                                                sigma_v=0.02):
    """calculate fermentation completenes and its uncertainty."""
    
    # Total ethanol produced
    TE = a * 10 * RHO_ETHANOL + v * MW_RATIO
    
    # Fermentation completeness
    FC = TE / (TE + s)
    
    # Partial derivatives
    dFC_da = s * 10 * RHO_ETHANOL / (TE + s)**2
    dFC_ds = -TE / (TE + s)**2
    dFC_dv = s * MW_RATIO / (TE + s)**2
    
    # Propagation of uncertainty (independent errors)
    sigma_FC = np.sqrt(
        (dFC_da * sigma_a)**2 + 
        (dFC_ds * sigma_s)**2 + 
        (dFC_dv * sigma_v)**2
    )
    
    return FC, sigma_FC

####################################################################################################


### Data Quality and Outlier Handling

The dataset originates from the official certification entity (CVRVV) of the 
Vinho Verde region, Portugal, where samples were tested between May 2004 and 
February 2007 using a computerized laboratory system (Cortez et al., 2009). 
This standardized collection procedure explains the absence of missing values 
and the overall consistency of the data.

Notably, the original authors performed **no outlier removal**. Their 
preprocessing was limited to reshaping the database (one row per wine sample), 
selecting the most common physicochemical tests explicitly "to avoid 
discarding examples," and standardizing features before modeling (Cortez et 
al., 2009). Quality scores were computed as the **median of at least three 
blind sensory assessors**, a robust aggregation that already mitigates 
extreme individual ratings.

Following the original methodology, we retained all samples and performed no 
outlier removal. Extreme values (e.g., volatile acidity > 1 g/L, residual 
sugar > 40 g/L) were verified to fall within physically plausible ranges for 
wine and were interpreted as rare but real styles or defects rather than 
measurement errors. This choice preserves the real-world variability that 
predictive models must ultimately handle.

In [ ]:
# Define physically plausible ranges for wine
physical_bounds = {
    'fixed acidity': (3, 16),           # g/L, tartaric equivalent
    'volatile acidity': (0.05, 2.0),    # g/L, acetic equivalent
    'citric acid': (0, 2.0),            # g/L
    'residual sugar': (0.5, 70),        # g/L
    'chlorides': (0.01, 0.8),           # g/L
    'free sulfur dioxide': (1, 80),     # mg/L
    'total sulfur dioxide': (5, 450),   # mg/L
    'density': (0.985, 1.050),          # g/mL
    'pH': (2.5, 4.5),
    'sulphates': (0.2, 2.5),            # g/L, K2SO4 equivalent
    'alcohol': (8, 16),                 # % vol
}

# Check for values outside physical bounds
outlier_report = []
for feature, (lower, upper) in physical_bounds.items():
    below = (df_raw[feature] < lower).sum()
    above = (df_raw[feature] > upper).sum()
    if below > 0 or above > 0:
        outlier_report.append({
            'Feature': feature,
            'Below Lower': below,
            'Above Upper': above,
            'Total Outliers': below + above,
            'Percentage': f"{(below + above) / len(df_raw) * 100:.2f}%"
        })

if outlier_report:
    print("Values outside physically plausible ranges:")
    print(pd.DataFrame(outlier_report))
else:
    print("All values fall within physically plausible ranges for wine.")

# Statistical outliers (beyond 1.5 IQR) — for documentation, not removal
print("\n" + "="*60)
print("Statistical outliers (beyond 1.5 IQR) — NOT removed:")
print("="*60)
numeric_cols = df_raw.select_dtypes(include=['float64', 'int64']).columns.tolist()
for feature in numeric_cols:
    Q1 = df_raw[feature].quantile(0.25)
    Q3 = df_raw[feature].quantile(0.75)
    IQR = Q3 - Q1
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = df_raw[(df_raw[feature] < lower_bound) | 
                      (df_raw[feature] > upper_bound)]
    
    if len(outliers) > 0:
        print(f"{feature}: {len(outliers)} outliers "
              f"({len(outliers)/len(df_raw)*100:.1f}%), "
              f"range [{df_raw[feature].min():.3f}, {df_raw[feature].max():.3f}]")



In [ ]:
# Boxplots with outliers highlighted
fig, axes = plt.subplots(3, 4, figsize=(18, 12))
axes = axes.flatten()

for ax, feature in zip(axes, numeric_cols):
    sns.boxplot(y=df_raw[feature], ax=ax, color='steelblue', 
                showfliers=True, flierprops={'marker': 'o', 'markersize': 3})
    pretty_feat = name_formatting.get(feature, feature)
    ax.set_title(pretty_feat, fontsize=10, fontweight='bold')
    ax.set_ylabel('')

plt.suptitle('Feature Distributions with Outliers Highlighted', 
             fontsize=14, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## 3.Red vs White: Fundamental Differences <a id='3'></a>

In [ ]:
#------------------3. Red vs White: Fundamental Differences---------------------#

numeric_cols.remove('quality')  # remove target

n_cols = len(numeric_cols)
n_grid_cols = 4
n_grid_rows = (n_cols + n_grid_cols - 1) // n_grid_cols

# dictionary for correct formating

fig, axes = plt.subplots(n_grid_rows, n_grid_cols, figsize=(18, 12))
axes = axes.flatten()

for i, col in enumerate(numeric_cols):
    sns.histplot(
        data=df_raw,
        x=col,
        hue='type',
        kde=True, 
        bins=30,
        ax=axes[i],
        palette={'red': 'crimson', 'white': 'goldenrod'},
        edgecolor='white',
        alpha=0.7,
        multiple='layer'
    )
    title = name_formatting.get(col, col.replace('_', ' ').title())
    axes[i].set_title(title, fontsize=12, fontweight='bold')
    axes[i].set_xlabel('')
    axes[i].tick_params(axis='both', labelsize=9)
    
    # Legend on first graph only
    if i == 0:
        axes[i].legend(title='Wine Type', loc='upper right')
    else:
        if axes[i].get_legend() is not None:
            axes[i].get_legend().remove()

# empty subplots deleted
for j in range(i + 1, len(axes)):
    fig.delaxes(axes[j])

plt.suptitle('Distribution of Wine Quality Features (Red vs White)', 
             fontsize=16, fontweight='bold', y=0.995)
plt.tight_layout()
plt.show()

## 4. Correlation Analysis <a id='4'></a>

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(20, 8))

for ax, wine_type in zip(axes, ['red', 'white']):
    subset = df_raw[df_raw['type'] == wine_type].drop(columns='type')
    corr = subset.corr(method='spearman')
    sns.heatmap(corr, annot=True, cmap='coolwarm', center=0, ax=ax, fmt='.2f',
                square=True, linewidths=0.5)
    ax.set_title(f'{wine_type.capitalize()} Wines: Spearman Correlations', 
                fontsize=13, fontweight='bold')

plt.tight_layout()
plt.show()


In [ ]:

# For each type: 
red_pairs = plot_correlated_pairs(
    df_raw[df_raw['type'] == 'red'], 
    'Red Wines', 
    threshold=0.3
)

white_pairs = plot_correlated_pairs(
    df_raw[df_raw['type'] == 'white'], 
    'White Wines', 
    threshold=0.3
)

## 5. Derived Features <a id='5'></a>

In [ ]:
# Let's build new dataframes with molar concentrations and see what new features can we get from  this
df = pd.DataFrame()
for column in df_raw.select_dtypes(include='number') :
    df[column] = molar_c(df_raw[column], column)
df['type']=df_raw['type']
df.describe()


In [ ]:
# Sulfur dioxide molecules that are bound to organic molecules
df['bound sulfur dioxide'] = df['total sulfur dioxide'] - df['free sulfur dioxide']

# Bound to free sulfur dioxide ratio
# Added little non-zero value to denominators to rule out division by zero just in case
df['sulfur dioxide binding'] = df['bound sulfur dioxide']/(df['free sulfur dioxide']+1e-6)

# Total acidity consists of volatile acidity and fixed acidity
# Note that citric acidity is already included in fixed acidity
df['total acidity'] = df['volatile acidity'] + df['fixed acidity']

# Volatile fraction in total acidity
df['volatile fraction'] = df['volatile acidity']/(df['total acidity']+1e-6)

# Alcohol to density ratio
df['alcohol to density ratio'] = df['alcohol']/(df['density']+1e-6)

# Sugar to alcohol ratio
df['sugar to alcohol ratio'] = df['residual sugar']/(df['alcohol']+1e-6)

# Fermentation completeness
MW_ETHANOL = 46.07
MW_ACETIC = 60.05
DENSITY_ETHANOL = 0.789  # g/mL

df['alcohol_g_per_L'] = df['alcohol'] * 10 * DENSITY_ETHANOL

ethanol_lost = df['volatile acidity'] * (MW_ETHANOL / MW_ACETIC)
total_ethanol = df['alcohol_g_per_L'] + ethanol_lost

df['fermentation completeness'] = (
    total_ethanol / (total_ethanol + df['residual sugar'] + 1e-6)
)

# Proton concentration in mol/L
df['H concentration']= 10**(-df['pH']) # mol/L

# Buffer capacity (relative value, see Discussion and Limitations sections)
df['buffer capacity'] = (
    (df['total acidity']) / 
    (df['H concentration'] + 1e-6)
)


## 6. Feature Distributions by Quality <a id='6'></a>

In [ ]:
# Dividing wines into quality groups
df['quality group'] = pd.cut(
    df['quality'], 
    bins=[2, 5, 7, 9], 
    labels=['Low', 'Medium', 'High']
)

top_features = ['fermentation completeness', 'buffer capacity',
                'sulfur dioxide binding', 'volatile fraction']

plot_quality_comparison(df, top_features)

# Call boxploting
plot_quality_comparison(df, top_features)

In [ ]:
quality_palette = {'Low': '#d73027', 'Medium': '#fee08b', 'High': '#1a9850'}

for wine_type in ['red', 'white']:
    df_subset = df[df['type'] == wine_type]
    
    g = sns.pairplot(
        df_subset,
        vars=top_features,
        hue='quality group',
        hue_order=['Low', 'Medium', 'High'],
        palette=quality_palette,
        diag_kind='kde',
        plot_kws={'alpha': 0.6, 's': 5},
        height=2.5,
        aspect=1
    )
    g.figure.suptitle(
        f'{wine_type.title()} Wines: Pairwise Distributions', 
        y=1.02, fontsize=16, fontweight='bold'
    )
    plt.show()

In [ ]:
# Parallel coordinates for each type of wine
features = ['fermentation completeness', 'buffer capacity',
            'sulfur dioxide binding', 'volatile fraction']

plot_parallel_coordinates(
    df[df['type'] == 'red'], 
    'Red Wines', 
    features
)

plot_parallel_coordinates(
    df[df['type'] == 'white'], 
    'White Wines', 
    features
)

## 7. Multivariate Exploration <a id='7'></a>

In [ ]:
for wine_type in ['red', 'white']:
    df_subset = df[df['type'] == wine_type].copy()
    
    df_clustered, kmeans, centroids = cluster_wines(
        df_subset, wine_type, features, n_clusters=4
    )
    
    plot_clusters_scatter(df_clustered, wine_type, features, n_clusters=4)
      
    plot_cluster_quality_distribution(df_clustered, wine_type, n_clusters=4)
    
    print(f"\n{wine_type.upper()} WINES - Cluster Centroids:")
    #print(centroids_original.round(3))
    print()

In [ ]:
for wine_type in ['red', 'white']:
    df_subset = df[df['type'] == wine_type].copy()
    
    # saving scaler
    scaler = StandardScaler()
    X = scaler.fit_transform(df_subset[features])
    
    kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
    df_subset['cluster'] = kmeans.fit_predict(X)
    
    # Centroids visualization
    centroids = plot_centroids_fixed(kmeans, scaler, features, wine_type, n_clusters=4)
    
    print(f"\n{wine_type.upper()} WINES - Cluster Centroids:")
    print(centroids.round(4))

In [ ]:


for wine_type in ['red', 'white']:
    df_subset = df[df['type'] == wine_type]
    X = StandardScaler().fit_transform(df_subset[features])
    
    print(f"\n{wine_type.upper()} WINES - Silhouette Scores:")
    for k in range(2, 8):
        km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
        score = silhouette_score(X, km.labels_)
        print(f"k={k}: silhouette = {score:.3f}")





In [ ]:
# clusters for red wine
df_red = df[df['type'] == 'red'].copy()
scaler_red = StandardScaler()
X_red = scaler_red.fit_transform(df_red[features])
kmeans_red = KMeans(n_clusters=4, random_state=42, n_init=10)
df_red['cluster_4'] = kmeans_red.fit_predict(X_red)

# for white wine
df_white = df[df['type'] == 'white'].copy()
scaler_white = StandardScaler()
X_white = scaler_white.fit_transform(df_white[features])
kmeans_white = KMeans(n_clusters=4, random_state=42, n_init=10)
df_white['cluster_4'] = kmeans_white.fit_predict(X_white)

# Group clusters
cluster_profiles = df_red.groupby('cluster_4').mean(numeric_only=True)
print(cluster_profiles.round(3))

# Mean values of all features
cluster_profiles = df_red.groupby('cluster_4').mean(numeric_only=True)
print(cluster_profiles.round(3))

# cluster_2 separated visually on scatterplots above and have a lot of residual sugar and SO2. These are likely sweet wines with early stopped fermentation


In [ ]:
# For white wines

# Mean values of all features
cluster_profiles = df_white.groupby('cluster_4').mean(numeric_only=True)
print(cluster_profiles.round(3))

In [ ]:
#features = ['fermentation completeness', 'buffer capacity',
#            'sulfur dioxide binding', 'volatile fraction']

# Red wines
#df_red = df[df['type'] == 'red'].copy()
profiles_red, quality_red = analyze_clusters(
    df_red, 'Red', 'cluster_4', features
)

# White wines
#df_white = df[df['type'] == 'white'].copy()
profiles_white, quality_white = analyze_clusters(
    df_white, 'White', 'cluster_4', features
)

In [ ]:
plot_all_features_by_cluster(df_red, 'Red Wines', 'cluster_4')
plot_all_features_by_cluster(df_white, 'White Wines', 'cluster_4')

In [ ]:
# Let's see if clusters differ in alcohol content
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(
    x='cluster_4', 
    y='alcohol', 
    data=df_red, 
    ax=axes[0],
    palette='viridis'
)
axes[0].set_title('Red Wines: Alcohol by Cluster', fontweight='bold')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Alcohol (% vol)')

for cluster_id in sorted(df_red['cluster_4'].unique()):
    subset = df_red[df_red['cluster_4'] == cluster_id]
    sns.kdeplot(
        subset['alcohol'], 
        label=f'Cluster {cluster_id}',
        ax=axes[1],
        alpha=0.7
    )
axes[1].set_title('Red Wines: Alcohol Distribution by Cluster', fontweight='bold')
axes[1].set_xlabel('Alcohol (% vol)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

sns.boxplot(
    x='cluster_4', 
    y='alcohol', 
    data=df_white, 
    ax=axes[0],
    palette='viridis'
)
axes[0].set_title('White Wines: Alcohol by Cluster', fontweight='bold')
axes[0].set_xlabel('Cluster')
axes[0].set_ylabel('Alcohol (% vol)')

for cluster_id in sorted(df_white['cluster_4'].unique()):
    subset = df_white[df_white['cluster_4'] == cluster_id]
    sns.kdeplot(
        subset['alcohol'], 
        label=f'Cluster {cluster_id}',
        ax=axes[1],
        alpha=0.7
    )
axes[1].set_title('White Wines: Alcohol Distribution by Cluster', fontweight='bold')
axes[1].set_xlabel('Alcohol (% vol)')
axes[1].set_ylabel('Density')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# See if alcohol distribution is multimodal
for wine_type in ['red', 'white']:
    subset = df[df['type'] == wine_type]['alcohol'].values.reshape(-1, 1)
    
    # Try different number of components
    bic_scores = []
    for n_comp in range(1, 6):
        gm = GaussianMixture(n_components=n_comp, random_state=42)
        gm.fit(subset)
        bic_scores.append(gm.bic(subset))
    
    # Optimal number is...
    optimal_k = np.argmin(bic_scores) + 1
    
    print(f"\n{wine_type.capitalize()} wines:")
    print(f"Optimal number of components: {optimal_k}")
    
    # Fit best model
    gm = GaussianMixture(n_components=optimal_k, random_state=42)
    gm.fit(subset)
    
    # Centers and weights
    centers = gm.means_.flatten()
    weights = gm.weights_
    
    print("Component centers and weights:")
    for i, (c, w) in enumerate(zip(sorted(centers), sorted(weights))):
        print(f"  Component {i}: center={c:.2f}% vol, weight={w:.3f}")

In [ ]:

# Use all original and derived features
# Now deal with two types separately not to confuse models
all_features = [c for c in df_red.columns if c not in ['quality', 'type', 'quality group']]
X = StandardScaler().fit_transform(df_red[all_features])
y = (df_red['quality'] >= 6).astype(int)

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=all_features).sort_values(ascending=False)
print(importances.head(10))
importances.head(10).plot(kind='barh', title='Top Feature Importances for Red Wines')
plt.tight_layout()
plt.show()

In [ ]:
all_features = [c for c in df_white.columns if c not in ['quality', 'type', 'quality group']]
X = StandardScaler().fit_transform(df_white[all_features])
y = (df_white['quality'] >= 6).astype(int)

rf = RandomForestClassifier(n_estimators=200, class_weight='balanced', random_state=42)
rf.fit(X, y)

importances = pd.Series(rf.feature_importances_, index=all_features).sort_values(ascending=False)
print(importances.head(10))
importances.head(10).plot(kind='barh', title='Top Feature Importances for White Wines')
plt.tight_layout()
plt.show()

In [ ]:
features = ['fermentation completeness', 'buffer capacity', 'chlorides', 'total acidity', 'citric acid', 
            'sulfur dioxide binding', 'volatile fraction', 'alcohol to density ratio', 'cluster_4']

X = df_red[features]
y = (df_red['quality'] >= 7).astype(int)  # 7+ is good

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

# Little tree
tree = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced')
tree.fit(X_train, y_train)

print(f"Train accuracy: {tree.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {tree.score(X_test, y_test):.3f}\n")

print(export_text(tree, feature_names=features))

plt.figure(figsize=(20, 8))
plot_tree(tree, feature_names=features, class_names=['Low', 'High'],
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree: Regions of High-Quality Red Wine', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
features = ['fermentation completeness', 'buffer capacity', 'chlorides', 'total acidity', 'citric acid', 
            'sulfur dioxide binding', 'volatile fraction', 'alcohol to density ratio', 'cluster_4']

X = df_white[features]
y = (df_white['quality'] >= 7).astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

tree = DecisionTreeClassifier(max_depth=3, random_state=42, class_weight='balanced')
tree.fit(X_train, y_train)

print(f"Train accuracy: {tree.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {tree.score(X_test, y_test):.3f}\n")

print(export_text(tree, feature_names=features))

plt.figure(figsize=(20, 8))
plot_tree(tree, feature_names=features, class_names=['Low', 'High'],
          filled=True, rounded=True, fontsize=10)
plt.title('Decision Tree: Regions of High-Quality White Wine', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Let's find collinear features
# Top10 from forest above for red wines
top10 = importances.head(10).index.tolist()
corr_matrix = df_red[top10].corr(method='spearman')
print('Collinear groups of features for red wines')
groups = collinear_groups(top10, corr_matrix, threshold=0.7)
for i, g in enumerate(groups):
    print(f"Group {i+1}: {g}")

# ...and for white ones
top10 = importances.head(10).index.tolist()
corr_matrix = df_white[top10].corr(method='spearman')
print('Collinear groups of features for white wines')
groups = collinear_groups(top10, corr_matrix, threshold=0.7)
for i, g in enumerate(groups):
    print(f"Group {i+1}: {g}")


In [ ]:
# There were too many features, some of them were collinear. Throw out some of them
representatives = ['alcohol to density ratio', 'volatile fraction', 'sulphates', 'sulfur dioxide binding', 'chlorides', 'total acidity', 'total sulfur dioxide', 'citric acid', 'fermentation completeness', 'buffer capacity']

X = df_red[features]
y = (df_red['quality'] >= 7).astype(int)  # 7+ is good
X_reduced = StandardScaler().fit_transform(df_red[representatives])
rf_reduced = RandomForestClassifier(n_estimators=200, random_state=42)
rf_reduced.fit(X_reduced, y)

imp = pd.Series(rf_reduced.feature_importances_, index=representatives).sort_values()
imp.plot(kind='barh', title='Feature Importance for Red Wines (collinearity-reduced)')
plt.tight_layout()
plt.show()

In [ ]:

rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_reduced, y)

perm = permutation_importance(rf, X_reduced, y, n_repeats=20, random_state=42)
perm_imp = pd.Series(perm.importances_mean, index=representatives).sort_values()
perm_imp.plot(kind='barh', title='Permutation Importance for Red Wines')
plt.tight_layout()
plt.show()

In [ ]:
X = df_white[features]
y = (df_white['quality'] >= 7).astype(int)  # 7+ is good
X_reduced = StandardScaler().fit_transform(df_white[representatives])
rf_reduced = RandomForestClassifier(n_estimators=200, random_state=42)
rf_reduced.fit(X_reduced, y)

imp = pd.Series(rf_reduced.feature_importances_, index=representatives).sort_values()
imp.plot(kind='barh', title='Feature Importance for Red Wines (collinearity-reduced)')
plt.tight_layout()
plt.show()

In [ ]:


rf = RandomForestClassifier(n_estimators=200, random_state=42).fit(X_reduced, y)

perm = permutation_importance(rf, X_reduced, y, n_repeats=20, random_state=42)
perm_imp = pd.Series(perm.importances_mean, index=representatives).sort_values()
perm_imp.plot(kind='barh', title='Permutation Importance for White Wines')
plt.tight_layout()
plt.show()

In [ ]:

df['FC_uncertainty'] = np.nan
for idx in df.index:
    fc, sigma_fc = fermentation_completeness_with_uncertainty(
        df.loc[idx, 'alcohol'],
        df.loc[idx, 'residual sugar'],
        df.loc[idx, 'volatile acidity']
    )
    df.loc[idx, 'FC_uncertainty'] = sigma_fc

print("Uncertainty of fermentation completeness:")
print(f"  Mean: {df['FC_uncertainty'].mean():.5f}")
print(f"  Median: {df['FC_uncertainty'].median():.5f}")
print(f"  Maximum: {df['FC_uncertainty'].max():.5f}")


fc_range = df['fermentation completeness'].max() - df['fermentation completeness'].min()
fc_std = df['fermentation completeness'].std()
fc_uncertainty_mean = df['FC_uncertainty'].mean()

print(f"\nRange of fermentation completeness in dataset: {fc_range:.5f}")
print(f"Standart deviation: {fc_std:.5f}")
print(f"Mean measurement uncertainty: {fc_uncertainty_mean:.5f}")
print(f"Ratio σ_measurement / σ_dataset: {fc_uncertainty_mean / fc_std:.2f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

df_sorted = df.sort_values('residual sugar')
axes[0].scatter(df_sorted['residual sugar'], df_sorted['FC_uncertainty'], 
               alpha=0.3, s=10, c='steelblue')
axes[0].set_xlabel('Residual Sugar (g/L)', fontsize=11)
axes[0].set_ylabel('σ(Fermentation Completeness)', fontsize=11)
axes[0].set_title('Uncertainty vs Residual Sugar', fontsize=12, fontweight='bold')
axes[0].axvline(x=4, color='red', linestyle='--', label='Dry/Sweet boundary')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

sugar_bins = pd.qcut(df['residual sugar'], q=10, duplicates='drop')
comparison = df.groupby(sugar_bins).agg({
    'fermentation completeness': ['mean', 'std'],
    'FC_uncertainty': 'mean'
}).round(5)

x = np.arange(len(comparison))
width = 0.35
axes[1].bar(x - width/2, comparison[('fermentation completeness', 'std')], 
           width, label='Observed std (dataset)', color='steelblue', alpha=0.7)
axes[1].bar(x + width/2, comparison[('FC_uncertainty', 'mean')], 
           width, label='Measurement uncertainty', color='crimson', alpha=0.7)
axes[1].set_xticks(x)
axes[1].set_xticklabels([f'{interval.mid:.1f}' for interval in comparison.index], 
                        rotation=45, fontsize=8)
axes[1].set_xlabel('Residual Sugar Bin (g/L)', fontsize=11)
axes[1].set_ylabel('Standard Deviation', fontsize=11)
axes[1].set_title('Observed Variability vs Measurement Uncertainty', 
                 fontsize=12, fontweight='bold')
axes[1].legend()
axes[1].grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# 8. Results and Discussion <a id='8'></a>

## Red vs White: Fundamental Differences and Similarities

Both red and white wines are produced from grapes through yeast fermentation. 
Notably, their alcohol distributions are remarkably similar, including local 
maxima. This shared distribution is determined by the physiological limit of 
yeast ethanol tolerance (~15–16% vol) and the target alcohol levels that 
winemakers aim to achieve across different wine styles.

However, significant differences exist between the two types. White wines, 
on average, exhibit higher acidity, more residual sugar, and higher sulfur 
dioxide content. These differences stem from distinct winemaking technologies: 
white wines are typically fermented without skin contact and require higher 
SO₂ levels for protection against oxidation.

As shown in subsequent sections, further similarities and differences are 
revealed in composition patterns and derived features.

---

## Correlation Analysis

The dataset contains numerous correlated feature pairs, most of which are 
physically determined (e.g., alcohol/density, fixed acidity/pH, free/total 
sulfur dioxide). Although most of these pairs do not involve quality, they 
should be kept in mind for feature engineering and modeling to avoid 
multicollinearity.

**Quality correlations:**

- **Alcohol:** The most prominent correlation with quality in both red and 
  white wines. This slight but notable positive correlation may be explained 
  by the perception that fuller-bodied (higher alcohol) wines are often rated 
  more favorably by tasters.

- **Volatile acidity:** Predominantly attributed to acetic acid, high values 
  indicate bacterial spoilage. Not surprisingly, there is a negative 
  correlation between volatile acidity and quality in red wines, though this 
  relationship is weaker or absent in white wines.

- **Sulphates:** A positive correlation between quality and sulphates is 
  observed in red wines but not in white wines. This may be explained by the 
  antimicrobial properties of sulfates, which inhibit spoilage organisms, or 
  by greater phenolic extraction in red winemaking.

- **Density and chlorides:** Weak negative correlations with quality in white 
  wines warrant further investigation. These may reflect subtle differences 
  in grape ripeness or winemaking practices.

**Key finding:** Correlation patterns differ between red and white wines, 
suggesting that quality determinants are type-specific.

---

## Feature Engineering

A number of derived features were created to capture chemically meaningful 
relationships in the data.

**Unit conversion:** The original composition values (g/L) were converted to 
molar concentrations (mol/L). Molar units are more natural for chemical 
analysis, as they allow direct comparison of different substances and 
facilitate the construction of stoichiometrically meaningful features.

**Derived features:**

1. **Bound sulfur dioxide** — the fraction of SO₂ bound to organic molecules, 
   primarily aldehydes and keto acids (oxidation products). Higher bound SO₂ 
   suggests greater oxidative stress and potentially lower wine quality.

2. **Sulfur dioxide binding** — the ratio of bound to free SO₂, characterizing 
   the equilibrium state of SO₂ binding. Higher values indicate more carbonyl 
   compounds available to bind SO₂, reflecting oxidative history.

3. **Total acidity** — the sum of volatile and fixed acidity. Note that citric 
   acid is already included in fixed acidity along with tartaric and other 
   non-volatile acids.

4. **Volatile fraction** — the ratio of volatile to total acidity. Higher 
   values indicate a greater proportion of acetic acid, suggesting microbial 
   spoilage.

5. **Alcohol to density ratio** and **sugar to alcohol ratio** — these capture 
   the balance between ethanol (which decreases density) and residual sugar 
   and other organoleptic compounds (which increase it).

6. **Fermentation completeness** — the extent to which sugar has been converted 
   to ethanol, accounting for concurrent volatile acid formation. This feature 
   distinguishes fully fermented (dry) wines from those with residual sugar.

7. **Hydrogen ion concentration (H⁺)** and relative **buffer capacity** — derived from 
   pH. Buffer capacity reflects the wine's ability to resist changes in acidity, 
   contributing to chemical stability.

**Note on fermentation completeness:** This feature is constrained to a narrow 
range (0.999–1.000) for dry wines, limiting its dynamic range. While it worked 
for exploratory analysis, alternative parameterizations (e.g., log-residual 
sugar) may be more suitable for predictive modeling.

---

## Multivariate Exploration

### Parallel Coordinates Analysis

The parallel coordinates plots reveal that wine quality cannot be separated 
by a simple linear combination of the four derived features. Lines from 
different quality groups overlap extensively, with no single axis showing 
clear stratification.

This suggests that:

- Quality is determined by **nonlinear interactions** between features, not by 
  any single dominant parameter.
- "Good" wine profiles are **diverse** — there are multiple chemical pathways 
  to high quality, whereas defects tend to converge on specific chemical 
  signatures.
- Linear visualization methods are insufficient to capture the quality boundary, 
  motivating the use of nonlinear models (decision trees, random forests) in 
  subsequent analysis.

### Cluster Analysis: Chemical Styles and Quality

The clustering analysis reveals four distinct chemical "styles" of wine, with 
remarkable consistency between red and white wines.

**1. "Clean Fermentation" Cluster (highest quality)**
- Low volatile fraction (minimal acetic acid)
- Complete fermentation
- Moderate buffer capacity
- Quality: 5.93 (red) / 6.05 (white)

This suggests that **quality is primarily determined by the absence of 
fermentation defects** rather than by any single positive attribute.

**2. "Oxidized/Spoiled" Cluster (lowest quality)**
- Low free SO₂ (insufficient protection)
- High bound SO₂ (extensive oxidation)
- High volatile fraction (bacterial spoilage)
- Quality: 5.39 (red) / 5.24 (white)

This represents wines with microbial contamination and oxidative damage, a 
universal defect across wine types.

**3. "Sweet/Stopped Fermentation" Cluster**
- High residual sugar
- Incomplete fermentation (0.999)
- High bound SO₂ (stabilization)
- Quality: 5.70 (red) / 5.69 (white)

This represents intentional sweet wine styles (semi-sweet, late harvest). 
Quality is average, confirming that sweetness is a stylistic choice, not a 
quality indicator.

**4. "Full-bodied" / "Acidic" Cluster**
- High buffer capacity
- Moderate volatile fraction
- Quality: intermediate

This represents wines with pronounced acidic structure.

### Key Insight

### Alcohol Distribution: Multimodality and Its Limits

The alcohol content distributions for both red and white wines show evidence 
of multimodality, with several local maxima. This pattern likely reflects 
different winemaking targets (light vs. full-bodied styles) and possibly 
different grape varieties or terroirs within the dataset.

However, when examining alcohol distributions within individual clusters, 
we observe that most clusters exhibit broad, overlapping distributions with 
multiple local maxima rather than distinct unimodal peaks. This suggests 
that while alcohol level contributes to wine style differentiation, it does 
not cleanly separate the chemical styles identified by clustering. The 
multimodality is more pronounced at the dataset level than within clusters, 
indicating that alcohol is one of several overlapping factors shaping wine 
composition rather than a primary cluster-defining variable.

This observation reinforces our earlier finding: chemical styles are 
multi-dimensional, and no single feature (including alcohol) can fully 
explain the cluster structure.

**Quality is orthogonal to chemical style.** A wine can be well-made (high 
quality) or poorly made (low quality) within any chemical style. The clusters 
capture winemaking regimes (clean vs. oxidized, dry vs. sweet), but quality 
depends on factors beyond these four features: balance, varietal expression, 
terroir, and winemaking skill.

### Alcohol Distribution and Hidden Structure

Despite the clustering being based on four derived features, the alcohol 
distribution shows evidence of multimodality, suggesting that the dataset 
contains distinct winemaking regimes not fully captured by these features. 
Local maxima in alcohol content may correspond to different target styles 
(light vs. full-bodied wines) or different grape varieties and terroirs.

Without varietal and regional data, we cannot definitively attribute these 
modes to specific factors. However, the multimodality suggests that **wine 
style is a latent variable** influencing both chemical composition and 
alcohol level.

---

## Feature Importance

### Handling Collinearity

As noted in the Correlation Analysis section, the dataset contains several 
highly correlated feature pairs (e.g., free/total sulfur dioxide, fixed 
acidity/pH, alcohol/density). Including all correlated features in a model 
can lead to:

- **Diluted importance scores:** Random Forest splits importance among 
  correlated features, making each appear less important than it truly is.
- **Redundant information:** Multiple features carry the same signal, 
  inflating model complexity without improving performance.
- **Interpretation ambiguity:** It becomes unclear which feature in a 
  correlated pair is truly driving predictions.

To address this, we identified groups of collinear features (Spearman 
correlation > 0.7) and selected one representative from each group based 
on chemical interpretability. The remaining features were used for Random 
Forest modeling.

### Permutation Importance Results

After removing redundant features, we computed permutation importance for 
the Random Forest classifier. This method measures how much model performance 
degrades when each feature's values are randomly shuffled, providing a robust 
estimate of feature contribution.

**Top features for red wines:**

| Feature | Permutation Importance |
|--------------------------|-------|
| alcohol to density ratio | 0.094 |
| sulphates                | 0.067 |
| volatile fraction        | 0.047 |
| total sulfur dioxide     | 0.023 |
| sulfur dioxide binding   | 0.020 |
| chlorides                | 0.015 |
| citric acid              | 0.015 |
| fermentation completeness| 0.010 |

**Top features for white wines:**

| Feature | Permutation Importance |
|--------------------------|-------|
| alcohol to density ratio | 0.152 |
| chlorides                | 0.061 |
| volatile fraction        | 0.047 |
| fermentation completeness| 0.044 |
| sulfur dioxide binding   | 0.043 |       
| buffer capacity          | 0.038 |
| total sulfur dioxide     | 0.029 |
| total acidity            | 0.015 |

### Interpretation

### Type-Specific Patterns: Red vs White Wines

The feature importance analysis reveals striking differences between red and 
white wines, reflecting their distinct chemistries and winemaking technologies.

#### 1. Alcohol to Density Ratio: Universal but Stronger for Whites

"Alcohol to density ratio" is the top predictor for both wine types, but its 
importance is notably higher for white wines (0.152) than for red wines 
(0.094). This likely reflects the greater stylistic diversity of white wines:

- White wines range from bone-dry to lusciously sweet, creating a wider 
  spectrum of alcohol-to-density relationships.
- Red wines are predominantly dry, reducing the variability of this feature.
- The balance between alcohol (body) and residual sugar/extract (density) is 
  a primary quality determinant in white wines, where freshness and balance 
  are paramount.

#### 2. Sulphates: A Red Wine Marker

Sulphates rank second for red wines (0.067) but do not appear in the white 
wine top features. This type-specific importance has several explanations:

- **Phenolic extraction:** Red wines are fermented with grape skins and 
  seeds, extracting minerals including sulfates alongside phenolics. Higher 
  sulphates may indicate more thorough extraction, which correlates with 
  fuller body and higher quality in red wines.

- **Antimicrobial protection:** Red wines have higher pH than whites and rely 
  more on sulphates and phenolics for microbial stability rather than SO₂ 
  alone.

- **Terroir signal:** In red winemaking, sulfate levels may more strongly 
  reflect vineyard soil composition and grape ripeness at harvest.

For white wines, the absence of skin contact means sulphate levels are lower 
and less variable, reducing their predictive value.

#### 3. Chlorides: A White Wine Marker

Chlorides rank second for white wines (0.061) but are much less important for 
red wines (0.015). This asymmetry reflects:

- **Taste sensitivity:** White wines are typically served chilled and have 
  lighter body, making them more sensitive to salty off-flavors. Chloride 
  variations are more perceptible and impactful on quality.

- **Microbial stability:** White wines have lower phenolic content and rely 
  more on SO₂ and other factors for stability. Chlorides contribute to 
  microbial inhibition, making them more critical in whites.

- **Lower baseline:** White wines generally have lower chloride concentrations 
  than reds, so variations are proportionally more significant.

#### 4. Derived Features: More Important for Whites

White wines show higher importance for chemically derived features:

- **Fermentation completeness** (0.044 vs 0.010): White wines span a wider 
  range of sweetness styles (bone-dry to sweet), making fermentation extent 
  more discriminative. Most red wines are fully fermented (dry), reducing 
  variability.

- **Buffer capacity** (0.038, absent for reds): Acid balance is paramount in 
  white wines, where freshness and crispness define quality. Buffer capacity 
  captures the wine's ability to maintain acidity, a key quality attribute 
  for whites.

- **Sulfur dioxide binding** (0.043 vs 0.020): White wines rely more heavily 
  on SO₂ for antioxidant protection (they lack the phenolic protection of 
  reds), so SO₂ binding dynamics are more quality-relevant.

For red wines, quality is more influenced by phenolic composition (tannins, 
anthocyanins) and structure, which are **not captured** in this dataset. This 
may explain why fewer derived features appear in the red wine top ranks.

#### 5. Citric Acid vs Total Acidity

Interestingly, citric acid appears in the red wine top features (0.015), while 
total acidity appears for whites (0.015). This subtle difference may reflect:

- Citric acid's role in malolactic fermentation (more common in reds)
- Total acidity's greater relevance for white wine freshness and balance

---

### Summary of Type-Specific Quality Determinants

| Quality Determinant | Red Wines | White Wines |
|---------------------|-----------|-------------|
| **Body/extract balance**    | Important | Very important |
| **Sulfate/mineral content** | Important | Less important |
| **Chloride/salt balance**   | Less important | Important |
| **Fermentation extent**     | Less important | Important |
| **Acid balance (buffer)**   | Less important | Important |
| **SO₂ management**          | Moderate | Very important |
| **Phenolic structure**      | Critical but not in dataset | Less relevant |

**Key insight:** Red and white wines are governed by **different quality 
determinants**. Models trained on combined data would miss these type-specific 
patterns. This reinforces the importance of **analyzing and modeling red and 
white wines separately**.

## Limitations

Several limitations of this analysis should be acknowledged:

1. **Approximate chemical features:** Some features in the dataset are 
   already approximations. Volatile and fixed acidity represent mixtures of 
   multiple acids but are reported as acetic acid and tartaric acid 
   equivalents, respectively. The determination of sulfates and sulfur dioxide 
   is subject to analytical errors and interferences.

2. **Narrow dynamic range:** The fermentation completeness feature is 
   constrained to a narrow range (0.999–1.000) for most wines, limiting its 
   discriminative power. Alternative parameterizations would be preferable 
   for predictive modeling.

3. **Missing contextual data:** The dataset lacks information about grape 
   variety, region, vintage, and winemaking techniques. These factors are 
   known to significantly influence wine quality and may confound the 
   chemical-quality relationship.

4. **Subjectivity of quality ratings:** Quality scores are based on sensory 
   evaluation by tasters, which introduces subjectivity and variability. 
   Different tasters may rate the same wine differently.

5. **Fundamental limitation:** It must be acknowledged that wine quality 
   cannot be unequivocally predicted from chemical composition alone, at 
   least not using the limited data presented in this dataset. Quality is a 
   multi-dimensional concept encompassing chemical, sensory, and contextual 
   factors.

### Uncertainty Analysis: Fermentation Completeness

The fermentation completeness feature was derived from three measured 
quantities (alcohol, residual sugar, volatile acidity), each subject to 
measurement uncertainty. Using first-order error propagation (total 
differential method), we estimate the uncertainty of this derived feature:

σ(FC) = √[ (∂FC/∂a)²σ_a² + (∂FC/∂s)²σ_s² + (∂FC/∂v)²σ_v² ]

Assuming typical measurement uncertainties for OIV-certified methods 
(σ_alcohol ≈ ±0.15% vol, σ_sugar ≈ ±0.3 g/L, σ_volatile acidity ≈ ±0.02 g/L):

- **For dry wines** (residual sugar < 4 g/L): σ(FC) ≈ 0.003. The observed 
  spread in fermentation completeness values (0.97–1.00) is partially 
  attributable to measurement error. Differences between wines with 
  FC = 0.995 and FC = 0.998 are **statistically indistinguishable**.

- **For sweet wines** (residual sugar > 10 g/L): σ(FC) ≈ 0.004. The 
  observed spread is much larger than measurement error, indicating genuine 
  variation in fermentation extent.

**Implication:** Fermentation completeness is a **reliable indicator only 
for sweet/semi-sweet wines**. For dry wines (the majority of the dataset), 
this feature is dominated by measurement noise and should be interpreted 
with caution. The residual sugar content itself is a more informative and 
directly measured alternative for distinguishing fermentation extent in dry 
wines.

---

In [ ]:
features_red = ['alcohol to density ratio', 'sulphates', 'volatile fraction', 
                'total sulfur dioxide', 'sulfur dioxide binding', 'chlorides', 
                'citric acid', 'fermentation completeness']
importance_red = [0.094, 0.067, 0.047, 0.023, 0.020, 0.015, 0.015, 0.010]

features_white = ['alcohol to density ratio', 'chlorides', 'volatile fraction',
                  'fermentation completeness', 'sulfur dioxide binding', 
                  'buffer capacity', 'total sulfur dioxide', 'total acidity']
importance_white = [0.152, 0.061, 0.047, 0.044, 0.043, 0.038, 0.029, 0.015]

# collect all the unique features
all_features = sorted(set(features_red + features_white))

# arrays with 0 for missing
imp_red = [importance_red[features_red.index(f)] if f in features_red else 0 
           for f in all_features]
imp_white = [importance_white[features_white.index(f)] if f in features_white else 0 
             for f in all_features]

# visual
x = np.arange(len(all_features))
width = 0.35

fig, ax = plt.subplots(figsize=(14, 8))
ax.barh(x - width/2, imp_red, width, label='Red Wines', color='crimson', alpha=0.8)
ax.barh(x + width/2, imp_white, width, label='White Wines', color='goldenrod', alpha=0.8)

ax.set_yticks(x)
ax.set_yticklabels(all_features)
ax.set_xlabel('Permutation Importance')
ax.set_title('Feature Importance: Red vs White Wines', fontsize=14, fontweight='bold')
ax.legend()
ax.grid(True, axis='x', alpha=0.3)

plt.tight_layout()
plt.show()

## Conclusions

This exploratory analysis of the UCI Wine Quality dataset builds upon the 
foundational work of Cortez et al. (2009), who demonstrated that 
physicochemical features can predict wine quality with accuracy within ±1 
point (on a 0-10 scale), making them valuable for decision support systems 
in oenology.

Our analysis extends this work by investigating the **chemical structure** 
underlying these predictions. Several key findings emerge:

### 1. Chemical Features Are Necessary but Not Sufficient

While Cortez et al. (2009) showed that chemical features can predict 
quality with useful accuracy, our analysis reveals that this prediction 
relies on a **small number of dominant features** (alcohol, volatile 
acidity, sulphates) and that quality is **not determined by a single 
chemical pathway**.

Clustering analysis shows that wines with similar chemical profiles can 
have different quality ratings, and wines with different profiles can 
achieve the same quality. This suggests that chemical features capture 
**necessary conditions** for quality (absence of defects, proper 
fermentation) but not **sufficient conditions** (balance, complexity, 
varietal expression).

### 2. Quality Control vs. Quality Grading

Our findings support the practical application proposed by Cortez et al. 
(2009): chemical analysis is well-suited for **quality control** 
(detecting defects, ensuring consistency) but has limitations for 
**quality grading** (predicting whether a wine will be rated 7 vs. 8).

For defect detection (quality < 5), chemical features are highly 
discriminative: high volatile acidity, low free SO₂, and signs of 
oxidation reliably identify spoiled wines. For fine quality distinctions, 
chemical features provide limited information, and sensory evaluation 
remains essential.

### 3. Derived Features Capture Meaningful Chemistry

Our feature engineering demonstrates that chemically motivated derived 
features (alcohol-to-density ratio, volatile fraction, fermentation 
completeness) can outperform or complement original features in importance. 
This validates the approach of using domain knowledge to enhance predictive 
models, as suggested by Cortez et al. (2009) when they note that "the 
relative importance of the inputs brought interesting insights regarding 
the impact of the analytical tests."

### 4. Type-Specific Patterns Require Separate Models

Cortez et al. (2009) correctly analyzed red and white wines separately due 
to their different taste profiles. Our analysis reinforces this: feature 
importance differs markedly between types (sulphates critical for reds, 
chlorides for whites), confirming that **separate models are necessary** 
for accurate prediction.

### Practical Implications

**For winemakers and quality control:** Following Cortez et al. (2009), 
chemical analysis can be integrated into decision support systems to:
- Flag wines with potential defects (high volatile acidity, low free SO₂)
- Ensure consistency across batches
- Guide process optimization by identifying controllable variables

However, our analysis suggests that such systems should be viewed as 
**assistive tools** rather than **replacement for sensory evaluation**. 
The model can suggest "this wine may have quality issues" but cannot 
definitively answer "is this a 7 or an 8?"

**For oenology education:** As Cortez et al. (2009) suggest, models can 
aid training by providing objective baselines. Students can compare their 
sensory assessments with model predictions, learning to calibrate their 
palates.

**For future research:** Integrating chemical data with sensory panels, 
spectroscopic measurements, and metadata (variety, terroir, vintage) could 
yield more robust quality models. However, the subjective nature of 
quality ratings remains a fundamental challenge.

### Broader Implications

This analysis underscores a fundamental principle: **wine quality is a 
multi-dimensional, context-dependent property**. Chemical features provide 
a valuable, objective foundation for quality assessment, but they capture 
only one dimension of a complex phenomenon that encompasses balance, 
complexity, typicity, and sensory harmony.

The data-driven approach proposed by Cortez et al. (2009) remains valid 
and useful, but it should be understood as **augmenting human expertise** 
rather than replacing it. The oenologist's palate, trained through 
experience and informed by chemical data, remains the ultimate arbiter of 
wine quality.